# Date Fruit Classification Using ANN

# 1. Importing paackages

In [126]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import seaborn as sns

# 2. Loading Dataset

In [127]:
df = pd.read_csv("dataset/datefruit_dataset.csv")

In [128]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   AREA           898 non-null    int64  
 1   PERIMETER      898 non-null    float64
 2   MAJOR_AXIS     898 non-null    float64
 3   MINOR_AXIS     898 non-null    float64
 4   ECCENTRICITY   898 non-null    float64
 5   EQDIASQ        898 non-null    float64
 6   SOLIDITY       898 non-null    float64
 7   CONVEX_AREA    898 non-null    int64  
 8   EXTENT         898 non-null    float64
 9   ASPECT_RATIO   898 non-null    float64
 10  ROUNDNESS      898 non-null    float64
 11  COMPACTNESS    898 non-null    float64
 12  SHAPEFACTOR_1  898 non-null    float64
 13  SHAPEFACTOR_2  898 non-null    float64
 14  SHAPEFACTOR_3  898 non-null    float64
 15  SHAPEFACTOR_4  898 non-null    float64
 16  MeanRR         898 non-null    float64
 17  MeanRG         898 non-null    float64
 18  MeanRB    

In [130]:
X = df.drop("Class", axis=1)
Y = df["Class"]

In [131]:
df["Class"].value_counts()

Class
DOKOL     204
SAFAVI    199
ROTANA    166
DEGLET     98
SOGAY      94
IRAQI      72
BERHI      65
Name: count, dtype: int64

# 3. Encoding The Dataset 

In [132]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
Y = le.fit_transform(Y)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 4. Sacling Dataset

In [133]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)   

# 5. ANN for model 

In [134]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.long)  
  
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.long)

In [135]:
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

In [136]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [137]:
# building the model

class DateFruitClassifier(nn.Module):
    def __init__(self):
        super(DateFruitClassifier, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.model(x)

In [138]:
model = DateFruitClassifier()

# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [139]:
# Training loop

epochs = 100

train_losses = []
test_losses = []

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad() # Reset gradients

        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()

        optimizer.step() # Update weights

        running_loss += loss
    
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss.item())

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}")


Epoch 1/100, Train Loss: 3.8669
Epoch 2/100, Train Loss: 3.0508
Epoch 3/100, Train Loss: 2.1861
Epoch 4/100, Train Loss: 1.6999
Epoch 5/100, Train Loss: 1.5015
Epoch 6/100, Train Loss: 1.2420
Epoch 7/100, Train Loss: 1.0807
Epoch 8/100, Train Loss: 0.9874
Epoch 9/100, Train Loss: 0.9296
Epoch 10/100, Train Loss: 0.8789
Epoch 11/100, Train Loss: 0.8476
Epoch 12/100, Train Loss: 0.8140
Epoch 13/100, Train Loss: 0.7758
Epoch 14/100, Train Loss: 0.7573
Epoch 15/100, Train Loss: 0.7465
Epoch 16/100, Train Loss: 0.7128
Epoch 17/100, Train Loss: 0.7035
Epoch 18/100, Train Loss: 0.7127
Epoch 19/100, Train Loss: 0.6869
Epoch 20/100, Train Loss: 0.6599
Epoch 21/100, Train Loss: 0.6533
Epoch 22/100, Train Loss: 0.6568
Epoch 23/100, Train Loss: 0.6620
Epoch 24/100, Train Loss: 0.6215
Epoch 25/100, Train Loss: 0.6262
Epoch 26/100, Train Loss: 0.6062
Epoch 27/100, Train Loss: 0.6081
Epoch 28/100, Train Loss: 0.5922
Epoch 29/100, Train Loss: 0.5908
Epoch 30/100, Train Loss: 0.5950
Epoch 31/100, Train

# 7. Evaluation (why every time u run the accuracy chnages ????)

In [140]:
# Evaluation 

model.eval()
correct = 0
total = 0

with torch.no_grad():


    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs.data, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("total", total)
print("correct", correct)

accuracy = (correct / total) * 100
print(f"Test Accuracy: {accuracy:.4f}")

total 180
correct 166
Test Accuracy: 92.2222


# 8. Saving Models

In [141]:
import pickle

# Save the trained model to a file
with open("model/datefruitANN.pkl", "wb") as file:  
    pickle.dump(model, file)

# 9. Makinf Requirement Files

In [142]:
import pkg_resources

# List of the packages you know you're using
required_packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'ipykernel',
    'torch',
]

requirements = []

for package in required_packages:
    try:
        version = pkg_resources.get_distribution(package).version
        requirements.append(f"{package}=={version}")
    except pkg_resources.DistributionNotFound:
        print(f"Package {package} not found in the environment.")

#requirements to a file
with open('requirements.txt', 'w') as f:
    for line in requirements:
        f.write(line + '\n')